# A. Webscraping de Datos Básicos con BeautifulSoup
Este notebook muestra cómo extraer la tabla de 'Datos Básicos' de la página de Gruplac usando Requests y BeautifulSoup, y convertirla en un DataFrame de pandas.

## 1. Instalación de Dependencias
Si no tienes instaladas las librerías necesarias, ejecuta:

In [ ]:
!pip install requests beautifulsoup4 lxml pandas

## 2. Importar Librerías

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

## 3. Descargar y Parsear la Página

In [2]:
url = "https://scienti.minciencias.gov.co/gruplac/jsp/visualiza/visualizagr.jsp?nro=00000000001840"
resp = requests.get(url)
resp.raise_for_status()
resp.encoding = resp.apparent_encoding
soup = BeautifulSoup(resp.text, "html.parser")

## 4. Localizar la Tabla de 'Datos Básicos'

In [3]:
for tbl in soup.find_all("table"):
    if tbl.find(string="Año y mes de formación"):
        tabla_datos = tbl
        break
else:
    raise RuntimeError("No encontré la tabla de Datos básicos")

## 5. Extraer Campos y Valores

In [7]:
datos = []
for fila in tabla_datos.find_all("tr"):
    celdas = fila.find_all(["th", "td"])
    if len(celdas) >= 2:
        campo = celdas[0].get_text(strip=True)
        valor = celdas[1].get_text(strip=True)
        datos.append((campo, valor))

## 6. Crear el DataFrame

In [ ]:
df = pd.DataFrame(datos, columns=["Campo", "Valor"])
df

## 7. Guardar en CSV (Opcional)

In [ ]:
df.to_csv("datos_basicos.csv", index=False, encoding="utf-8-sig")

# B. Webscraping de Datos Básicos con BeautifulSoup (Múltiples URLs)
Este notebook muestra cómo extraer la tabla de 'Datos Básicos' de varias páginas de Gruplac, convertir cada resultado en un DataFrame pivoteado (un registro por URL) y concatenarlos en una tabla final.

## 2. Importar Librerías

## 3. Definir Función de Scraping y Pivot

In [ ]:
def scrape_datos_basicos(url):
    resp = requests.get(url)
    resp.raise_for_status()
    resp.encoding = resp.apparent_encoding
    soup = BeautifulSoup(resp.text, "html.parser")

    # Localizar tabla de Datos Básicos
    tabla = None
    for tbl in soup.find_all("table"):
        if tbl.find(string="Año y mes de formación"):
            tabla = tbl
            break
    if tabla is None:
        raise RuntimeError(f"No encontré la tabla de Datos básicos en {url}")

    # Extraer pares campo-valor
    datos = []
    for fila in tabla.find_all("tr"):
        celdas = fila.find_all(["th", "td"])
        if len(celdas) >= 2:
            campo = celdas[0].get_text(strip=True)
            valor = celdas[1].get_text(strip=True)
            datos.append((campo, valor))

    # DataFrame y pivot
    df = pd.DataFrame(datos, columns=["Campo", "Valor"])
    pivot_df = df.set_index("Campo").T
    pivot_df.insert(0, "url", url)  # añadir columna con URL
    return pivot_df

## 4. Listado de URLs a Procesar

In [ ]:
urls = [
    "https://scienti.minciencias.gov.co/gruplac/jsp/visualiza/visualizagr.jsp?nro=00000000001840",
    "https://scienti.minciencias.gov.co/gruplac/jsp/visualiza/visualizagr.jsp?nro=00000000002515",
    "https://scienti.minciencias.gov.co/gruplac/jsp/visualiza/visualizagr.jsp?nro=00000000006944",
]

## 5. Ejecutar Scraping en Lote y Concatenar Resultados

In [ ]:
# Recorrer cada URL y recolectar DataFrames pivoteados
lista_dfs = []
for url in urls:
    try:
        df_piv = scrape_datos_basicos(url)
        lista_dfs.append(df_piv)
    except Exception as e:
        print(f"Error en {url}: {e}")

# Concatenar todos los registros en una sola tabla
df_final = pd.concat(lista_dfs, ignore_index=True)
df_final

## 6. (Opcional) Guardar Resultado en CSV

In [ ]:
df_final.to_csv("datos_basicos_lote.csv", index=False, encoding="utf-8-sig")